# ViT-LoRA trên DS1 và DS2

**Thái Văn Huy · 23130068 · Tuần 3**

| Thành phần | Cấu hình |
|---|---|
| Backbone | ViT-Base/16 pretrained ImageNet-21k, đầu vào 224 × 224 |
| LoRA | r=8, α=16; query/value của 12 tầng attention |
| DS1 | Phân loại nhị phân; 5-fold CV với validation độc lập trong mỗi fold |
| DS2 | Phân loại bốn lớp và hồi quy bounding box từ segmentation mask |
| Chọn checkpoint | Validation total loss nhỏ nhất |
| Threshold Moving | Chọn ngưỡng 0.35–0.40 trên validation; đối chiếu test với ngưỡng 0.50 |
| Định vị DS2 | Mean Box IoU và Pointing Game Accuracy trên segmentation test |

Mục tiêu đánh giá là Recall có u >96%. Notebook chưa chứa kết quả thực nghiệm MRI.


## 1. Runtime và cấu hình

Notebook hỗ trợ GPU Colab và GPU máy Windows qua **Connect to a local runtime**.
Chế độ `local` dùng lại source, dữ liệu đã chuẩn bị và checkpoint; mặc định tiếp tục DS2.
`RUN_DIR` là thư mục checkpoint/kết quả. `DRIVE_SYNC_DIR` là thư mục đích `Huy/tuan3`
trong Google Drive for desktop, nếu đã thiết lập đồng bộ trên Windows.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, platform

RUNTIME_MODE = "local" if os.name == "nt" else "colab"
LOCAL_PROJECT_DIR = Path(r"C:\Users\Admin\Downloads\BrainTumorDetect_Huy_week3\BrainTumorDetect")
DRIVE_SYNC_DIR = None  # Local: Path(r"duong_dan_Google_Drive\My Drive\Huy\tuan3")

PROJECT_DIR = LOCAL_PROJECT_DIR if RUNTIME_MODE == "local" else Path("/content/BrainTumorDetect")
RUN_DIR = PROJECT_DIR / "runs/huy_week3" if RUNTIME_MODE == "local" else Path("/content/drive/MyDrive/Huy/tuan3")
SOURCE_MODE = "existing" if RUNTIME_MODE == "local" else "archive"
DATA_MODE = "existing" if RUNTIME_MODE == "local" else "kaggle"
REPO_URL = "https://github.com/MikanQuyt/BrainTumorDetect.git"
REPO_REF = "thai_van_huy_tuan3"
NAVONEEL_ZIP = ""
INDK214_ZIP = ""
MAX_EPOCHS = 1000
EARLY_STOPPING_PATIENCE = 25
BATCH_SIZE = 8
NUM_WORKERS = 0 if RUNTIME_MODE == "local" else 2
DS1_FOLD = None
RUN_DS1 = RUNTIME_MODE != "local"
RUN_DS2 = True
RUN_OFFLINE_TESTS = RUNTIME_MODE != "local"
PREPARE_DATA = RUNTIME_MODE != "local"
INSTALL_REQUIREMENTS = RUNTIME_MODE != "local"
REQUIRE_DS2_CHECKPOINT = RUNTIME_MODE == "local"
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
print("Runtime:", RUNTIME_MODE, "| Python:", sys.executable)

In [ ]:
_synced_files = {}

def sync_outputs():
    if RUNTIME_MODE != "local" or DRIVE_SYNC_DIR is None:
        return
    for source in sorted(RUN_DIR.rglob("*")):
        if not source.is_file() or source.suffix not in {".pth", ".json", ".csv", ".png", ".txt"}:
            continue
        relative = source.relative_to(RUN_DIR)
        stat = source.stat()
        signature = (stat.st_mtime_ns, stat.st_size)
        if _synced_files.get(relative) == signature:
            continue
        destination = Path(DRIVE_SYNC_DIR) / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(f".{destination.name}.{os.getpid()}.tmp")
        try:
            shutil.copy2(source, temporary)
            current = source.stat()
            if signature != (current.st_mtime_ns, current.st_size):
                continue
            os.replace(temporary, destination)
            _synced_files[relative] = signature
        finally:
            temporary.unlink(missing_ok=True)


def try_sync_outputs():
    try:
        sync_outputs()
    except OSError as error:
        print(f"Drive sync pending: {error}. Checkpoint goc van o {RUN_DIR}")


def run_command(command, *, cwd=None, log_name=None):
    log = None
    if log_name is not None:
        log_dir = RUN_DIR / "logs"
        log_dir.mkdir(parents=True, exist_ok=True)
        log = (log_dir / Path(log_name).name).open("a", encoding="utf-8")
        log.write("\nCOMMAND: " + " ".join(map(str, command)) + "\n")
        log.flush()
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    try:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                              errors="replace", bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    if log is not None:
                        log.write(line)
                        log.flush()
                    if " patience=" in line or line.startswith("RESUME "):
                        try_sync_outputs()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if status:
            raise subprocess.CalledProcessError(status, command)
    finally:
        if log is not None:
            log.close()
        try_sync_outputs()

In [ ]:
if RUNTIME_MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    if not RUN_DIR.parent.is_dir():
        raise FileNotFoundError(f"Missing Drive project directory: {RUN_DIR.parent}")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
elif RUNTIME_MODE == "local":
    if not (PROJECT_DIR / "engine/train_vit_lora_huy.py").is_file():
        raise FileNotFoundError(f"Sai LOCAL_PROJECT_DIR: {PROJECT_DIR}")
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(f"Thieu thu muc checkpoint hien tai: {RUN_DIR}")
    if DRIVE_SYNC_DIR is not None:
        DRIVE_SYNC_DIR = Path(DRIVE_SYNC_DIR).resolve()
        local = RUN_DIR.resolve()
        if not DRIVE_SYNC_DIR.is_dir():
            raise FileNotFoundError(f"Drive for desktop chua co thu muc: {DRIVE_SYNC_DIR}")
        if (DRIVE_SYNC_DIR == local or DRIVE_SYNC_DIR.is_relative_to(local)
                or local.is_relative_to(DRIVE_SYNC_DIR)):
            raise ValueError("RUN_DIR va DRIVE_SYNC_DIR phai la hai thu muc tach biet")
    else:
        print("Drive sync: chua cau hinh. Checkpoint dang luu tren may.")
else:
    raise ValueError("RUNTIME_MODE phai la local hoac colab")
print("Checkpoint va ket qua:", RUN_DIR)
if DRIVE_SYNC_DIR is not None and RUNTIME_MODE == "local":
    print("Dong bo Drive sau moi epoch da luu:", DRIVE_SYNC_DIR)

## 2. Source và môi trường

`local`: dùng nguyên source và môi trường đã cài. `colab`: nạp source archive hoặc Git
và cài requirements; giữ nguyên cặp torch/torchvision CUDA của runtime.

In [ ]:
if SOURCE_MODE == "existing":
    pass
elif SOURCE_MODE == "archive":
    source_dir = RUN_DIR / "source"
    source_dir.mkdir(exist_ok=True)
    archive = source_dir / "BrainTumorDetect_Huy_week3.zip"
    if not archive.is_file():
        from google.colab import files
        uploaded = files.upload()
        expected = "BrainTumorDetect_Huy_week3.zip"
        if expected not in uploaded:
            raise ValueError(f"Expected source archive: {expected}")
        archive.write_bytes(uploaded[expected])
        del uploaded
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            target = (PROJECT_DIR.parent / entry.filename).resolve()
            if not target.is_relative_to(PROJECT_DIR.resolve()):
                raise ValueError("Gói source phải chỉ chứa thư mục BrainTumorDetect/")
        z.extractall(PROJECT_DIR.parent)
elif SOURCE_MODE == "github":
    if not (PROJECT_DIR / ".git").exists():
        run_command(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT_DIR)])
    else:
        run_command(["git", "fetch", "origin", REPO_REF], cwd=PROJECT_DIR)
        run_command(["git", "switch", REPO_REF], cwd=PROJECT_DIR)
        run_command(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=PROJECT_DIR)
else:
    raise ValueError("SOURCE_MODE phai la existing, archive hoac github")

required = ["engine/train_vit_lora_huy.py", "data/prepare_huy_week3.py",
            "reports/tuan2/vit_baseline/ds1/splits.csv", "reports/tuan2/vit_baseline/ds2/splits.csv"]
for name in required:
    if not (PROJECT_DIR / name).is_file():
        raise FileNotFoundError(f"Thiếu {name}. Kiểm tra gói source/nhánh Git.")
os.chdir(PROJECT_DIR)
print("Source sẵn sàng:", PROJECT_DIR)

In [ ]:
if INSTALL_REQUIREMENTS:
    run_command([sys.executable, "-m", "pip", "install", "-q", "-r",
                 "experiments/requirements-huy-week3.txt"])

check = "import torch, json, sys, platform; " \
        "assert torch.cuda.is_available(), 'CUDA unavailable'; " \
        "print(json.dumps(dict(python=sys.version, executable=sys.executable, " \
        "platform=platform.platform(), torch=torch.__version__, " \
        "cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0))))"
env = dict(os.environ, PYTHONIOENCODING="utf-8")
runtime_info = json.loads(subprocess.check_output([sys.executable, "-c", check],
                            text=True, encoding="utf-8", env=env))
print("Torch:", runtime_info["torch"], "| CUDA:", runtime_info["cuda"], "| GPU:", runtime_info["gpu"])
environment_dir = RUN_DIR / "environment"
environment_dir.mkdir(exist_ok=True)
(environment_dir / f"{RUNTIME_MODE}_runtime.json").write_text(
    json.dumps(runtime_info, ensure_ascii=False, indent=2), encoding="utf-8")
(environment_dir / f"pip-freeze-{RUNTIME_MODE}.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"],
                            text=True, encoding="utf-8", env=env), encoding="utf-8")

## 3. Kiểm tra pipeline

Bộ kiểm tra dùng dữ liệu tổng hợp và ViT nhỏ để kiểm tra split, bảo toàn mask, threshold, early stopping, checkpoint resume và Auto-Prompt. Kết quả này xác minh logic chương trình, không thay thế đánh giá trên MRI thật.


In [ ]:
if RUN_OFFLINE_TESTS:
    run_command([sys.executable, "-m", "unittest", "tests.test_huy_week3", "-v"], log_name="pipeline_tests.log")


## 4. Dữ liệu

DS1: 253 ảnh. DS2: 7.023 ảnh classification và 2.192 cặp ảnh–mask.
Chế độ `local` dùng lại `dataset/huy_week3/` đã chuẩn bị. Chế độ `colab` hỗ trợ Kaggle
hoặc ZIP dataset trên Drive.

In [ ]:
raw = PROJECT_DIR / "dataset"
raw.mkdir(exist_ok=True)

def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            if not (destination / entry.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Đường dẫn ZIP không hợp lệ")
        z.extractall(destination)
    return destination

if DATA_MODE == "existing":
    for name in ("audit.json", "splits.json", "ds1_annotations.json", "ds2_annotations.json"):
        if not (raw / "huy_week3" / name).is_file():
            raise FileNotFoundError(f"Thieu du lieu da chuan bi: {raw / 'huy_week3' / name}")
    print("Dung lai du lieu:", raw / "huy_week3")
else:
    if DATA_MODE == "kaggle":
        import kagglehub
        nav_source = Path(kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection"))
        ind_source = Path(kagglehub.dataset_download("indk214/brain-tumor-dataset-segmentation-and-classification"))
    elif DATA_MODE == "drive_zip":
        if not Path(NAVONEEL_ZIP).is_file() or not Path(INDK214_ZIP).is_file():
            raise FileNotFoundError("Điền đúng hai đường dẫn ZIP trên Drive")
        nav_source = safe_extract(NAVONEEL_ZIP, Path("/content/raw_navoneel"))
        ind_source = safe_extract(INDK214_ZIP, Path("/content/raw_indk214"))
    else:
        raise ValueError("DATA_MODE không hợp lệ")

    nav_candidates = [nav_source] + sorted(p.parent for p in nav_source.rglob("yes") if p.is_dir())
    nav_base = next((p for p in nav_candidates if (p/"yes").is_dir() and (p/"no").is_dir()), None)
    ind_candidates = [ind_source/"DATASET"] + sorted(p for p in ind_source.rglob("DATASET") if p.is_dir())
    ind_base = next((p for p in ind_candidates if (p/"classification").is_dir() and (p/"Segmentation").is_dir()), None)
    if nav_base is None or ind_base is None:
        raise ValueError("Bố cục dataset khác tuần 2; kiểm tra thư mục trước khi tiếp tục")
    for label in ("no", "yes"):
        shutil.copytree(nav_base/label, raw/"navoneel"/label, dirs_exist_ok=True)
    shutil.copytree(ind_base, raw/"indk214"/"DATASET", dirs_exist_ok=True)
    print("Đã lấy dữ liệu gốc. Ô tiếp theo kiểm tra số lượng, hash và split.")

## 5. Tiền xử lý và chia tập

DS1 giữ nguyên các fold tuần 2. Với DS2, xác minh split classification cũ theo ảnh gốc rồi mở rộng nhóm trùng bằng ảnh sau crop/CLAHE và liên kết qua segmentation. Nhóm nối nhiều tập cũ được gộp về một tập theo ưu tiên **test > validation > train**; mọi ảnh test cũ vẫn nằm trong test. Kiểm tra lại toàn bộ nhóm sau khi sửa split.

`audit.json` ghi kích thước split trước/sau và số ảnh chuyển; `ds2_split_changes.csv` lưu từng mẫu chuyển tập. Split DS2 sau sửa có thể khác tuần 2, cần ghi rõ khi so sánh kết quả.

Mask được crop đồng bộ với MRI và kiểm tra bảo toàn pixel u. Tên PNG đầu ra giữ cả đuôi gốc để phân biệt `Y92.jpg` và `Y92.png`. Kiểm tra trùng chưa bao gồm ảnh gần trùng hoặc tính độc lập ở cấp bệnh nhân.


In [ ]:
if PREPARE_DATA:
    run_command([sys.executable, "-m", "data.prepare_huy_week3"], log_name="prepare_data.log")
audit = json.loads((PROJECT_DIR / "dataset/huy_week3/audit.json").read_text(encoding="utf-8"))
for dataset in ("ds1", "ds2"):
    print(dataset, "images:", audit[dataset]["n"], "masks:", audit[dataset]["n_masks"])
for source, destination in (("audit.json", "data_audit.json"),
                            ("splits.json", "data_splits.json"),
                            ("ds2_split_changes.csv", "ds2_split_changes.csv")):
    shutil.copy2(PROJECT_DIR / "dataset/huy_week3" / source, RUN_DIR / destination)

## 6. Huấn luyện

Logic huấn luyện nằm trong `engine/train_vit_lora_huy.py`, sử dụng `models/vit_lora.py`, DataLoader và các hàm loss của repo.

- AdamW cập nhật LoRA và các head; backbone gốc đóng băng.
- Tối đa 1.000 epoch; early stopping theo validation loss với patience=25.
- ReduceLROnPlateau giảm learning rate khi validation loss không cải thiện.
- `loss_best.pth` lưu LoRA và các head tại epoch có validation loss thấp nhất.
- `loss_latest.pth` lưu thêm optimizer, scheduler, AMP scaler, epoch và trạng thái resume.

Ngưỡng được chọn sau khi nạp best checkpoint, chỉ dùng validation classification. Tập test không tham gia chọn checkpoint hoặc ngưỡng.


In [ ]:
if RUNTIME_MODE == "local":
    import psutil
    for process in psutil.process_iter(["pid", "cmdline", "cwd"]):
        try:
            command = process.info["cmdline"] or []
            if "engine.train_vit_lora_huy" not in command or "--out" not in command:
                continue
            active_out = Path(command[command.index("--out") + 1])
            if not active_out.is_absolute():
                if not process.info["cwd"]:
                    continue
                active_out = Path(process.info["cwd"]) / active_out
            if active_out.resolve() == RUN_DIR.resolve():
                raise RuntimeError(f"CMD van dang train (PID {process.pid}). Dung luot CMD truoc khi chay o nay.")
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue

if RUN_DS2 and REQUIRE_DS2_CHECKPOINT:
    latest = RUN_DIR / "ds2/fold_01/loss_latest.pth"
    if not latest.is_file():
        raise FileNotFoundError(f"Thieu checkpoint de resume: {latest}")
    check = "import torch, sys; p=torch.load(sys.argv[1], map_location='cpu', weights_only=False); " \
            "print('Checkpoint epoch:', p['epoch'], '| Next epoch:', p['epoch']+1)"
    run_command([sys.executable, "-c", check, str(latest)])
try_sync_outputs()

base_command = [sys.executable, "-m", "engine.train_vit_lora_huy",
                "--out", str(RUN_DIR), "--epochs", str(MAX_EPOCHS),
                "--patience", str(EARLY_STOPPING_PATIENCE),
                "--batch-size", str(BATCH_SIZE), "--workers", str(NUM_WORKERS),
                "--device", "cuda", "--amp", "--resume", "auto"]
if RUN_DS1:
    command = base_command + ["--dataset", "ds1"]
    if DS1_FOLD is not None:
        command += ["--fold", str(DS1_FOLD)]
    run_command(command, log_name="train_ds1.log")

In [ ]:
if RUN_DS2:
    run_command(base_command + ["--dataset", "ds2"], log_name="train_ds2.log")


## 7. Kết quả đánh giá

**DS1:** `ds1/cv_summary.json` tổng hợp 253 dự đoán outer-test sau khi hoàn tất năm fold. Mỗi fold có ngưỡng được chọn từ validation riêng.

**DS2:** classification test và segmentation test được báo cáo riêng. `localization_summary.json` ghi Mean Box IoU và Pointing Game Accuracy; `localization_predictions.csv` lưu từng mẫu. Pointing Game tính điểm đỉnh dự đoán nằm trong mask u thật. Chỉ số toàn pipeline tính ảnh có u bị detector bỏ sót là một lần thất bại.

Báo cả Recall, Specificity và FP/FN ở ngưỡng 0.50 và ngưỡng được chọn. Kết quả chưa đạt mục tiêu 96% vẫn được giữ nguyên. So sánh với tuần 2 cần ghi rõ khác biệt ViT-Tiny/112 và ViT-Base/224.


In [ ]:
summaries = [("DS1", Path("ds1/cv_summary.json")),
             ("DS2", Path("ds2/fold_01/test_metrics.json"))]
for label, relative in summaries:
    candidates = [RUN_DIR / relative]
    if RUNTIME_MODE == "local" and DRIVE_SYNC_DIR is not None:
        candidates.append(Path(DRIVE_SYNC_DIR) / relative)
    path = next((p for p in candidates if p.is_file()), None)
    if path is not None:
        print("\n", label, path)
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    else:
        print(label, "chua co bao cao tai cac duong dan:", *candidates)

from IPython.display import display, Image
for path in sorted(RUN_DIR.glob("ds*/fold_*/learning_curves.png")):
    print(path.parent)
    display(Image(filename=str(path)))


## Khôi phục và bàn giao

Chỉ chạy một tiến trình huấn luyện vào cùng `RUN_DIR`. Khi chuyển từ CMD, đợi dòng
`E.../1000 train=... val=... best=... patience=...` rồi nhấn Ctrl+C. Checkpoint trước dòng
này đã được lưu hoàn chỉnh; notebook tiếp tục từ epoch kế tiếp.

Chế độ local không gọi `drive.mount()`. Nếu `DRIVE_SYNC_DIR` trỏ tới đúng `Huy/tuan3`
trong Google Drive for desktop, notebook tự sao chép checkpoint và kết quả sau mỗi epoch
đã lưu; ứng dụng Drive chịu trách nhiệm tải lên cloud. Nếu chưa cấu hình, kết quả chỉ
lưu ở `RUN_DIR`. Giữ ứng dụng Drive chạy và kiểm tra trạng thái đồng bộ khi bàn giao.

Notebook và đầu ra các ô có thể lưu trên Drive qua giao diện Colab; đó là việc riêng với
đồng bộ file `.pth`. Thông tin Python, PyTorch, GPU và phiên bản thư viện được ghi vào
`environment/` trong thư mục kết quả.

Giữ nguyên source, cấu hình và dữ liệu khi resume. Trainer kiểm tra fingerprint và
khôi phục optimizer, scheduler, AMP scaler cùng checkpoint mới nhất. Checkpoint gồm
LoRA và các head, cần đúng pretrained backbone khi nạp lại. Không thay scheduler hoặc
thêm warm-up giữa lượt chạy. Dataset, checkpoint và `runs/` không đưa vào Git.